# Stage 1 — CTC validation with Stage 0 maintenance control

This notebook tests whether the A/B/C phase and communication configurations differ when all conditions use the same Stage 0 setting. It adds workspace re-entry as a **controlled factor**: 0 pA (off) versus 50 pA (the Stage 0 candidate). Re-entry is applied identically within each A/B/C comparison, so it cannot be mistaken for a CTC-specific effect.

A: reciprocal, phase-gated FLN condition. B: shared rhythm with long-range E→E rate transmission disabled. C: one-way phase-drag control. We record stimulus-period phase locking (R), rate mutual information (MI), ignition, and post-stimulus persistence. Because re-entry itself is a positive-feedback extension, a result at 50 pA validates the combined system only; evidence for CTC requires A to outperform controls at the **same** re-entry level.

The original Stage 1 pilot files are preserved. This notebook writes to new `stage1_ctc_reentry_factor_*` output files. The initial 4-seed pilot is complete. This expanded paired confirmation uses 24 seeds (144 model runs); results are still exploratory pending a larger study.


## Step 1 — fix the Stage 0 settings and 24-seed confirmation conditions

Run duration is extended to 1600 ms so each trial includes 550 ms after the V4 stimulus ends. Keep the Stage 0 stimulus, FLN scale, K, phase-gate depth, and ignition criterion fixed. Seeds 1–24 are paired across all six condition combinations (A/B/C × re-entry off/on).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'model_class.py').exists()), None)
if project_root is None:
    raise RuntimeError('Run Jupyter from the repository or a subdirectory.')
experiment_dir = project_root / 'project_experiment'
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(experiment_dir))
from stage0_two_node import DEFAULTS, simulate_stage0
from ctc_validation import _mutual_information_bits

replication_K = [10.8]
replication_drive = 0.5  # 500 pA
N_SEEDS = 24
replication_seeds = np.arange(1, N_SEEDS + 1)  # paired seed confirmation
MI_SURROGATES = 10
RESUME_EXISTING = True
phase_gate = 1.0
workspace_reentry_levels_pA = [0.0, 50.0]
IGNITION_THRESHOLD_HZ = 20.0
IGNITION_DURATION_MS = 50.0
stage1_params = {
    **DEFAULTS,
    'stimulus_drive': replication_drive,
    'stimulus_current_pA': 500.0,
    'stim_on_ms': 50.0,
    'stim_off_ms': 1050.0,
    'duration_ms': 1600.0,
    'g_sw': 0.95,
    'phase_gate_strength': phase_gate,
    'sensory_area': 'V4',
    'workspace_areas': ['F1', 'F2', 'F3', 'F4', '7a', '7b'],
    'ignition_threshold_hz': IGNITION_THRESHOLD_HZ,
}
print('Stage 0 protocol: V4 500 pA, 50–1050 ms; run to 1600 ms; K=10.8; phase gate=1.0')
print(f'24-seed confirmation factorial: A/B/C × workspace re-entry {workspace_reentry_levels_pA} pA × seeds {replication_seeds.tolist()}')
print(f'Total confirmation runs: {3 * len(workspace_reentry_levels_pA) * N_SEEDS}; ignition = >=20 Hz for >=50 ms')


## Step 2 — calculate synchrony, information, ignition, and persistence

R and rate MI are measured during the stimulus. MI uses a circular-shift null to reduce bias from autocorrelation; it remains an association measure, not proof of directed communication. Ignition is evaluated over the full run, while post-stimulus ignition is checked only after 1050 ms.


In [ ]:
def _mi_with_shift_null(r_s, r_w, seed, bins=6, n_surrogates=30):
    raw = _mutual_information_bits(r_s, r_w, bins=bins)
    if len(r_s) < 40 or n_surrogates <= 0:
        return raw, 0.0, raw
    rng = np.random.default_rng(int(seed) + 500_000)
    low = max(10, int(0.1 * len(r_s)))
    high = max(low + 1, int(0.9 * len(r_s)))
    shifts = rng.integers(low, high, size=n_surrogates)
    null = np.array([_mutual_information_bits(r_s, np.roll(r_w, int(s)), bins=bins) for s in shifts])
    bias = float(null.mean())
    return raw, bias, max(0.0, raw - bias)


def _max_true_run(mask):
    padded = np.r_[False, np.asarray(mask, dtype=bool), False]
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    return int(np.max(edges[1::2] - edges[::2])) if len(edges) else 0


def _record_trial(result, condition, K, seed, reentry_pA, stim_on, stim_off, n_surrogates=10):
    time_ms = result['time_ms']
    during = (time_ms >= stim_on) & (time_ms < stim_off)
    after = time_ms >= stim_off
    dt_ms = float(np.median(np.diff(time_ms)))
    phase_delta = result['theta_w_rad'][during] - result['theta_s_rad'][during]
    R = float(abs(np.exp(1j * phase_delta).mean()))
    mi_raw, mi_null, mi_corrected = _mi_with_shift_null(
        result['r_s_hz'][during], result['r_w_hz'][during], seed, n_surrogates=n_surrogates
    )
    post_duration_ms = _max_true_run(result['r_w_hz'][after] >= IGNITION_THRESHOLD_HZ) * dt_ms
    return {
        'condition': condition, 'K_rad_s': float(K), 'seed': int(seed),
        'workspace_reentry_current_pA': float(reentry_pA),
        'stimulus_pA': 500.0, 'stim_on_ms': float(stim_on), 'stim_off_ms': float(stim_off),
        'run_duration_ms': float(time_ms[-1] - time_ms[0]),
        'R_PLV_during_stim': R,
        'MI_rates_raw_bits': mi_raw, 'MI_shift_null_bits': mi_null,
        'MI_rates_corrected_bits': mi_corrected,
        'W_ignited_20Hz_50ms': bool(result['ignited']),
        'W_latency_ms': result['ignition_latency_ms'],
        'W_poststim_ignited_20Hz_50ms': bool(post_duration_ms >= IGNITION_DURATION_MS),
        'W_max_poststim_above20_ms': post_duration_ms,
        'mean_rS_during_hz': float(result['r_s_hz'][during].mean()),
        'mean_rW_during_hz': float(result['r_w_hz'][during].mean()),
        'mean_rW_after_hz': float(result['r_w_hz'][after].mean()),
        'max_rW_hz': float(result['r_w_hz'].max()),
        'max_single_workspace_area_hz': float(result['workspace_area_rates_hz'].max()),
        'mean_gSW_during': float(result['g_sw_effective'][during].mean()),
    }


## Step 3 — run or resume the 24-seed paired confirmation

This creates a separate 24-seed output file; the 4-seed pilot and earlier Stage 1 results remain unchanged. The CSV is checkpointed after each seed block; rerunning the cell resumes matching conditions.


In [ ]:
conditions = [
    ('A', 'A_dynamic_gate', phase_gate),
    ('B', 'B_common_drive_no_link', 0.0),
    ('C', 'C_one_way_phase_drag', phase_gate),
]
trials_path = experiment_dir / 'stage1_ctc_reentry_factor_24seed_trials.csv'
trial_rows = []
completed = set()
if RESUME_EXISTING and trials_path.exists():
    old = pd.read_csv(trials_path)
    valid = (
        old['K_rad_s'].isin(replication_K)
        & old['seed'].isin(replication_seeds)
        & old['workspace_reentry_current_pA'].isin(workspace_reentry_levels_pA)
        & old['condition'].isin([c[1] for c in conditions])
        & old['stimulus_pA'].eq(500.0)
        & old['stim_on_ms'].eq(50.0)
        & old['stim_off_ms'].eq(1050.0)
        & old['run_duration_ms'].ge(1599.0)
    )
    old = old.loc[valid].drop_duplicates(
        ['workspace_reentry_current_pA', 'K_rad_s', 'seed', 'condition'], keep='last'
    )
    trial_rows.extend(old.to_dict('records'))
    completed = {
        (float(r['workspace_reentry_current_pA']), float(r['K_rad_s']), int(r['seed']), str(r['condition']))
        for r in trial_rows
    }
    print(f'Resuming {len(completed)} matching trials.')

total_runs = len(workspace_reentry_levels_pA) * len(replication_K) * len(replication_seeds) * len(conditions)
try:
    from tqdm.auto import tqdm
    progress = tqdm(total=total_runs, initial=len(completed), desc='Stage 1 paired factorial scan', unit='run')
except ImportError:
    progress = None

for reentry_pA in workspace_reentry_levels_pA:
    for K in replication_K:
        for seed in replication_seeds:
            for config, label, gate in conditions:
                key = (float(reentry_pA), float(K), int(seed), label)
                if key in completed:
                    continue
                params = {
                    **stage1_params,
                    'phase_coupling_rad_s': float(K),
                    'phase_gate_strength': float(gate),
                    'workspace_reentry_current_pA': float(reentry_pA),
                }
                result = simulate_stage0(config, seed=int(seed), params=params)
                row = _record_trial(
                    result, label, K, int(seed), reentry_pA,
                    stage1_params['stim_on_ms'], stage1_params['stim_off_ms'],
                    n_surrogates=MI_SURROGATES,
                )
                trial_rows.append(row)
                completed.add(key)
                if progress is not None:
                    progress.update(1)
            pd.DataFrame(trial_rows).to_csv(trials_path, index=False)
if progress is not None:
    progress.close()

stage1_trials = pd.DataFrame(trial_rows).drop_duplicates(
    ['workspace_reentry_current_pA', 'K_rad_s', 'seed', 'condition'], keep='last'
)
stage1_trials.to_csv(trials_path, index=False)
expected_rows = len(workspace_reentry_levels_pA) * len(replication_K) * len(replication_seeds) * len(conditions)
assert len(stage1_trials) == expected_rows, f'Incomplete scan: expected {expected_rows} rows, found {len(stage1_trials)}'
assert not stage1_trials.duplicated(['workspace_reentry_current_pA', 'K_rad_s', 'seed', 'condition']).any()
print(f'Saved complete {len(stage1_trials)}-trial scan to {trials_path}')


## Step 4 — summarize ignition and test paired condition differences

Compare A against B and C separately within each re-entry level, pairing by seed. Report exact McNemar/binomial tests for binary ignition outcomes. This asks whether the reciprocal phase-gated condition adds an advantage over the controls after holding workspace re-entry fixed.


In [ ]:
group_cols = ['workspace_reentry_current_pA', 'condition']
summary = stage1_trials.groupby(group_cols).agg(
    n=('seed', 'nunique'),
    ignition_probability=('W_ignited_20Hz_50ms', 'mean'),
    poststim_ignition_probability=('W_poststim_ignited_20Hz_50ms', 'mean'),
    mean_R=('R_PLV_during_stim', 'mean'),
    mean_MI_corrected=('MI_rates_corrected_bits', 'mean'),
    mean_rW_during_hz=('mean_rW_during_hz', 'mean'),
    mean_rW_after_hz=('mean_rW_after_hz', 'mean'),
    max_single_area_rate_hz=('max_single_workspace_area_hz', 'max'),
).reset_index()
# Wilson 95% intervals for ignition and post-stimulus persistence proportions.
z = 1.96
for metric, prefix in [('ignition_probability', 'ignition'), ('poststim_ignition_probability', 'poststim')]:
    n = summary['n'].to_numpy(dtype=float)
    q = summary[metric].to_numpy(dtype=float)
    den = 1 + z*z/n
    center = (q + z*z/(2*n)) / den
    half = z*np.sqrt(q*(1-q)/n + z*z/(4*n*n)) / den
    summary[f'{prefix}_wilson95_low'] = center - half
    summary[f'{prefix}_wilson95_high'] = center + half
display(summary.round(3))

contrast_rows = []
for reentry_pA in workspace_reentry_levels_pA:
    block = stage1_trials[stage1_trials.workspace_reentry_current_pA == reentry_pA]
    for control in ['B_common_drive_no_link', 'C_one_way_phase_drag']:
        a = block[block.condition == 'A_dynamic_gate'].set_index('seed')
        b = block[block.condition == control].set_index('seed')
        common = a.index.intersection(b.index)
        a_ign = a.loc[common, 'W_ignited_20Hz_50ms'].astype(bool).to_numpy()
        b_ign = b.loc[common, 'W_ignited_20Hz_50ms'].astype(bool).to_numpy()
        a_only = int(np.sum(a_ign & ~b_ign))
        b_only = int(np.sum(~a_ign & b_ign))
        discordant = a_only + b_only
        p = float(binomtest(min(a_only, b_only), discordant, 0.5).pvalue) if discordant else 1.0
        contrast_rows.append({
            'workspace_reentry_current_pA': reentry_pA,
            'comparison': f'A_dynamic_gate vs {control}',
            'n_paired_seeds': len(common),
            'A_only_ignited': a_only,
            'control_only_ignited': b_only,
            'paired_ignition_risk_difference': float(a_ign.mean() - b_ign.mean()),
            'mcnemar_exact_two_sided_p': p,
            'A_mean_R': float(a.loc[common, 'R_PLV_during_stim'].mean()),
            'control_mean_R': float(b.loc[common, 'R_PLV_during_stim'].mean()),
            'A_mean_MI_corrected_bits': float(a.loc[common, 'MI_rates_corrected_bits'].mean()),
            'control_mean_MI_corrected_bits': float(b.loc[common, 'MI_rates_corrected_bits'].mean()),
        })
contrasts = pd.DataFrame(contrast_rows)
contrasts.to_csv(experiment_dir / 'stage1_ctc_reentry_factor_24seed_contrasts.csv', index=False)
display(contrasts.round(4))
summary.to_csv(experiment_dir / 'stage1_ctc_reentry_factor_24seed_summary.csv', index=False)


## Step 5 — visualize the CTC comparison at each maintenance level

The two panels separate re-entry off and on. Interpret A/B/C differences only within a panel; differences between panels show the contribution of the added maintenance feedback. A high ignition rate shared by all conditions is not evidence for CTC-specific benefit.


In [ ]:
condition_order = ['A_dynamic_gate', 'B_common_drive_no_link', 'C_one_way_phase_drag']
condition_labels = {'A_dynamic_gate': 'A: reciprocal gate',
                    'B_common_drive_no_link': 'B: common drive/no link',
                    'C_one_way_phase_drag': 'C: one-way phase drag'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, reentry_pA in zip(axes, workspace_reentry_levels_pA):
    block = stage1_trials[stage1_trials.workspace_reentry_current_pA == reentry_pA]
    g = block.groupby('condition').agg(
        n=('seed', 'nunique'), ignition=('W_ignited_20Hz_50ms', 'mean'),
        poststim=('W_poststim_ignited_20Hz_50ms', 'mean'),
        R=('R_PLV_during_stim', 'mean'), MI=('MI_rates_corrected_bits', 'mean'),
    ).reindex(condition_order)
    x = np.arange(len(condition_order))
    y = g['ignition'].to_numpy()
    n = g['n'].to_numpy(dtype=float)
    z = 1.96; den = 1 + z*z/n
    center = (y + z*z/(2*n)) / den
    half = 1.96*np.sqrt(y*(1-y)/n + z*z/(4*n*n)) / den
    ax.bar(x, y, color=['#4C78A8', '#F58518', '#54A24B'], alpha=.88)
    ax.errorbar(x, y, yerr=np.vstack([y-(center-half), (center+half)-y]), fmt='none', ecolor='black', capsize=4)
    ax.set_xticks(x, [condition_labels[k] for k in condition_order], rotation=18, ha='right')
    ax.set_ylim(0, 1.08); ax.set_title(f'Workspace re-entry: {reentry_pA:g} pA')
    ax.grid(axis='y', alpha=.25)
axes[0].set_ylabel('Ignition probability (Wilson 95% CI)')
fig.suptitle('Stage 1: CTC controls with maintenance feedback held fixed')
fig.tight_layout()
fig.savefig(experiment_dir / 'stage1_ctc_reentry_factor_24seed_ignition.png', dpi=180, bbox_inches='tight')
plt.show()

measurement_summary = stage1_trials.groupby(['workspace_reentry_current_pA', 'condition']).agg(
    mean_R=('R_PLV_during_stim', 'mean'), mean_MI_bits=('MI_rates_corrected_bits', 'mean'),
    poststim_ignition=('W_poststim_ignited_20Hz_50ms', 'mean'),
).reset_index()
display(measurement_summary.round(3))


### Interpretation checklist

- First confirm each row has 24 unique paired seeds and no missing condition cells.
- If A exceeds B and C at the same re-entry level in ignition and corrected MI, that supports an additional reciprocal CTC contribution in this model.
- If all three conditions ignite similarly at 50 pA, the maintenance loop explains the ignition increase, while Stage 1 has not shown a CTC advantage.
- If 0 pA still reproduces the earlier rare-ignition pattern, that is consistent with Stage 0. Keep the 20 Hz / 50 ms rule fixed when interpreting outcomes.


### Preliminary pilot result (4 paired seeds)

The 24-run pilot completed for seeds 1, 6, 12, and 23. With re-entry off, ignition was A 2/4, B 0/4, C 2/4; post-stimulus sustained ignition was A 0/4, B 0/4, C 2/4. With 50 pA re-entry, ignition and post-stimulus sustained ignition were A 4/4, B 0/4, C 4/4. Thus, re-entry raised A and C to 100% in this small sample, but did not make the reciprocal A condition outperform the one-way C control.

At 50 pA, mean PLV was 0.759 for A and 0.982 for C; corrected rate MI was 0.064 bits for A and 0.071 bits for C. The paired A-versus-C ignition test was not significant (exact p=1.0, with no discordant seeds). A-versus-B was 4/4 versus 0/4 (exact p=0.125), which is suggestive but too small to establish a reliable difference. Treat these as pilot observations only: four seeds cannot confirm a CTC-specific effect, and the five-shift MI estimate is intentionally coarse.

Pilot outputs: `stage1_ctc_reentry_factor_pilot_trials.csv`, `stage1_ctc_reentry_factor_pilot_summary.csv`, `stage1_ctc_reentry_factor_pilot_contrasts.csv`, and `stage1_ctc_reentry_factor_pilot_ignition.png`.

### 24-seed confirmation result

The expanded scan completed 144 trials (24 paired seeds in each of six groups). Results:

| Workspace re-entry | A: reciprocal dynamic gate | B: common rhythm, no rate link | C: one-way phase drag |
|---|---:|---:|---:|
| 0 pA: ignition | 2/24 (8.3%) | 0/24 | 18/24 (75.0%) |
| 0 pA: post-stimulus ignition | 0/24 | 0/24 | 15/24 (62.5%) |
| 50 pA: ignition | 24/24 (100%) | 0/24 | 23/24 (95.8%) |
| 50 pA: post-stimulus ignition | 24/24 (100%) | 0/24 | 23/24 (95.8%) |

The 50 pA loop raises A from 8.3% to 100% ignition and maintains activity in all 24 seeds. C is also high (95.8%) under 50 pA, so there is no demonstrated reciprocal-CTC advantage over the one-way control there (paired exact p=1.0). With re-entry off, C ignites more often than A (75.0% vs 8.3%; paired exact p=0.000145). C also has higher mean stimulus PLV (0.980 vs 0.845); corrected MI is similar or somewhat higher for C (0.093 vs 0.074 bits without re-entry; 0.076 vs 0.075 bits with re-entry). These findings do not support the claim that reciprocal phase coupling outperforms the one-way control at this Stage 0 setting.

B has high PLV despite almost zero workspace rate and no ignition. This is a useful reminder that synchrony alone is not sufficient evidence of rate transfer. The result is specific to these A/B/C implementations and the 24-seed parameter set; the 50 pA re-entry remains an explicit added positive-feedback mechanism, not an original AToum parameter.

Files: `stage1_ctc_reentry_factor_24seed_trials.csv`, `stage1_ctc_reentry_factor_24seed_summary.csv`, `stage1_ctc_reentry_factor_24seed_contrasts.csv`, and `stage1_ctc_reentry_factor_24seed_ignition.png`.